In [3]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [7]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv", encoding="latin-1")

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


In [6]:
import os

print(os.getcwd())
print(os.listdir())

/Users/caiomaia/ironhack-workspace/labs/week-03/W3D4-lab-natural-language-processing/your-code
['main.ipynb']


### Let's divide the training and test set into two partitions

In [8]:
# Your code

from sklearn.model_selection import train_test_split

# Split the original dataset into two partitions:
# - 80% of the rows will go to data_train
# - 20% of the rows will go to data_val

data_train, data_val = train_test_split(
    data,
    test_size=0.2, ## Reserve 20% of the data for validation.
    random_state=42 # Keep the split reproducible every time we run the notebook.
)

# Check the size of each partition
print("Training set shape", data_train.shape)
print("Validation set shape:", data_val.shape)

Training set shape (800, 2)
Validation set shape: (200, 2)


In [ ]:
# training set
# → used to teach the model

# validation set
# → used to evaluate how well it generalizes to unseen data

## Data Preprocessing

In [9]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [11]:
# Show the names of the columns in the training dataset.
print("Columns:", data_train.columns.tolist())

# Show the first few rows so we can identify
# which column contains the email text
# and which column contains the spam/ham label.
display(data_train.head())

Columns: ['text', 'label']


,text,label
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1
535,I have not been able to reach oscar this am. W...,0
695,; Huma Abedin B6I'm checking with Pat on the 5...,0
557,I can have it announced here on Monday - can't...,0
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1


In [12]:
# Import Python's regular-expression module.
# We will use it to search for patterns inside text.
import re


# Create a function that removes HTML-related content from one text string.
def remove_html(text):

    # Remove <script>...</script> blocks.
    # re.DOTALL allows the pattern to continue across multiple lines.
    text = re.sub(
        r"<script.*?>.*?</script>",
        " ",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    # Remove <style>...</style> blocks containing CSS.
    text = re.sub(
        r"<style.*?>.*?</style>",
        " ",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    # Remove HTML comments such as:
    # <!-- this is a comment -->
    text = re.sub(
        r"<!--.*?-->",
        " ",
        text,
        flags=re.DOTALL
    )

    # Remove the remaining HTML tags such as:
    # <p>, </p>, <div>, <br>, etc.
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Return the cleaned text.
    return text


# Apply the HTML-cleaning function to the text column
# of both the training and validation datasets.
data_train["preprocessed_text"] = data_train["text"].apply(remove_html)
data_val["preprocessed_text"] = data_val["text"].apply(remove_html)


# Show an example so we can inspect the result.
print(data_train["preprocessed_text"].iloc[0])

----------- REGARDS, MR NELSON SMITH.KINDLY REPLY ME ON MY PRIVATE EMAIL ADDRESS;nelsonsmith2000@yahoo.com


In [ ]:
# Your code

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [13]:
# Your code

# Create a function that performs the next stage of text cleaning.
def clean_text(text):

    # Remove special characters and keep only letters and spaces.
    # Example: "hello!!!" -> "hello"
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove numbers.
    # Example: "win 1000 dollars" -> "win dollars"
    text = re.sub(r"\d+", " ", text)

    # Remove single-character words.
    # Example: "a message x here" -> "message here"
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)

    # Remove a prefixed "b" that sometimes appears
    # when text was originally stored as bytes.
    # Example: "b hello world" -> "hello world"
    text = re.sub(r"^b\s+", "", text)

    # Replace multiple spaces with a single space.
    # Example: "hello     world" -> "hello world"
    text = re.sub(r"\s+", " ", text)

    # Convert everything to lowercase.
    # Example: "FREE MONEY" -> "free money"
    text = text.lower()

    # Remove spaces from the beginning and end.
    text = text.strip()

    # Return the cleaned version.
    return text


# Apply the cleaning function to the training data.
data_train["preprocessed_text"] = (
    data_train["preprocessed_text"]
    .apply(clean_text)
)

# Apply the same cleaning function to the validation data.
data_val["preprocessed_text"] = (
    data_val["preprocessed_text"]
    .apply(clean_text)
)

# Show one cleaned example.
print(data_train["preprocessed_text"].iloc[0])

regards mr nelson smith kindly reply me on my private email address nelsonsmith yahoo com


## Now let's work on removing stopwords
Remove the stopwords.

In [14]:
# Your code

# Create a set of common English stopwords.
# A set is useful here because checking membership is fast.
stop_words = set(stopwords.words("english"))

# Create a function that removes stopwords from one text string.
def remove_stopwords(text):

    # Split the text into individual words.
    words = text.split()

    # Keep only words that are NOT in the stopword list.
    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Join the remaining words back into one string.
    return " ".join(filtered_words)


# Apply stopword removal to the training dataset.
data_train["preprocessed_text"] = (
    data_train["preprocessed_text"]
    .apply(remove_stopwords)
)

# Apply the same stopword removal to the validation dataset.
data_val["preprocessed_text"] = (
    data_val["preprocessed_text"]
    .apply(remove_stopwords)
)

# Show one example after stopword removal.
print(data_train["preprocessed_text"].iloc[0])

regards mr nelson smith kindly reply private email address nelsonsmith yahoo com


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [15]:
# Your code
# Import NLTK's lemmatizer.
# A lemmatizer tries to reduce words to their base dictionary form.
from nltk.stem import WordNetLemmatizer

# Create the lemmatizer object that we will reuse.
lemmatizer = WordNetLemmatizer()


# Create a function that lemmatizes one text string.
def lemmatize_text(text):

    # Split the sentence into individual words.
    words = text.split()

    # Lemmatize each word one by one.
    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Join the words back into one string.
    return " ".join(lemmatized_words)


# Apply lemmatization to the training dataset.
data_train["preprocessed_text"] = (
    data_train["preprocessed_text"]
    .apply(lemmatize_text)
)

# Apply the same lemmatization to the validation dataset.
data_val["preprocessed_text"] = (
    data_val["preprocessed_text"]
    .apply(lemmatize_text)
)

# Show one example after lemmatization.
print(data_train["preprocessed_text"].iloc[0])


regard mr nelson smith kindly reply private email address nelsonsmith yahoo com


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [16]:
# Your code

# Import CountVectorizer.
# It converts text into word-count features.
from sklearn.feature_extraction.text import CountVectorizer


# Create a CountVectorizer object.
# This will learn the vocabulary from the text.
count_vectorizer = CountVectorizer()


# -----------------------------
# TOP WORDS IN HAM MESSAGES
# -----------------------------

# Select only the preprocessed text from HAM messages.
# Here we assume label 0 = ham.
ham_text = data_train[
    data_train["label"] == 0
]["preprocessed_text"]

# Learn the ham vocabulary and count how often each word appears.
ham_matrix = count_vectorizer.fit_transform(ham_text)

# Add the counts for each word across all ham messages.
ham_word_counts = ham_matrix.sum(axis=0)

# Get the vocabulary words in the same column order as the matrix.
ham_words = count_vectorizer.get_feature_names_out()

# Combine each word with its total count.
ham_frequency = list(
    zip(
        ham_words,
        ham_word_counts.A1
    )
)

# Sort from most frequent to least frequent.
ham_top_10 = sorted(
    ham_frequency,
    key=lambda x: x[1],
    reverse=True
)[:10]


# -----------------------------
# TOP WORDS IN SPAM MESSAGES
# -----------------------------

# Create a fresh vectorizer for the spam text.
spam_vectorizer = CountVectorizer()

# Select only the preprocessed text from SPAM messages.
# Here we assume label 1 = spam.
spam_text = data_train[
    data_train["label"] == 1
]["preprocessed_text"]

# Learn the spam vocabulary and count word occurrences.
spam_matrix = spam_vectorizer.fit_transform(spam_text)

# Add the counts for each word across all spam messages.
spam_word_counts = spam_matrix.sum(axis=0)

# Get the vocabulary words.
spam_words = spam_vectorizer.get_feature_names_out()

# Combine words with their total counts.
spam_frequency = list(
    zip(
        spam_words,
        spam_word_counts.A1
    )
)

# Sort from most frequent to least frequent.
spam_top_10 = sorted(
    spam_frequency,
    key=lambda x: x[1],
    reverse=True
)[:10]


# Print the results.
print("Top 10 HAM words:")
print(ham_top_10)

print("\nTop 10 SPAM words:")
print(spam_top_10)

Top 10 HAM words:
[('state', np.int64(117)), ('pm', np.int64(97)), ('would', np.int64(94)), ('mr', np.int64(89)), ('president', np.int64(89)), ('time', np.int64(81)), ('percent', np.int64(80)), ('obama', np.int64(77)), ('call', np.int64(74)), ('secretary', np.int64(74))]

Top 10 SPAM words:
[('money', np.int64(847)), ('account', np.int64(743)), ('bank', np.int64(646)), ('fund', np.int64(626)), ('transaction', np.int64(471)), ('business', np.int64(424)), ('mr', np.int64(423)), ('country', np.int64(422)), ('million', np.int64(370)), ('company', np.int64(366))]


## Extra features

In [17]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,regard mr nelson smith kindly reply private em...,0,0,79
535,I have not been able to reach oscar this am. W...,0,able reach oscar supposed send pdb receive,0,0,42
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,huma abedin checking pat work jack jake rest a...,0,0,76
557,I can have it announced here on Monday - can't...,0,announced monday today,0,0,22
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,bank africaagence san pedro bp san pedro cote ...,1,1,1050


## How would work the Bag of Words with Count Vectorizer concept?

In [18]:
# Your code

# Import CountVectorizer.
# This converts text into numerical word-count features.
from sklearn.feature_extraction.text import CountVectorizer

# Create the Bag of Words vectorizer.
bow_vectorizer = CountVectorizer()

# Learn the vocabulary from the TRAINING text
# and convert the training messages into word-count vectors.
X_train_bow = bow_vectorizer.fit_transform(
    data_train["preprocessed_text"]
)

# Convert the VALIDATION text using the SAME vocabulary
# learned from the training set.
X_val_bow = bow_vectorizer.transform(
    data_val["preprocessed_text"]
)

# Store the labels separately.
# These are the correct answers the model will learn from.
y_train = data_train["label"]
y_val = data_val["label"]

# Check the size of the resulting matrices.
print("Training BoW shape:", X_train_bow.shape)
print("Validation BoW shape:", X_val_bow.shape)

Training BoW shape: (800, 28042)
Validation BoW shape: (200, 28042)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [19]:
# Your code

# Create a TF-IDF vectorizer.
# TF-IDF converts text into numerical features,
# but gives more weight to informative words
# and less weight to very common words.
tfidf_vectorizer = TfidfVectorizer()

# Learn the vocabulary and IDF weights from the TRAINING data,
# then convert the training text into TF-IDF features.
X_train_tfidf = tfidf_vectorizer.fit_transform(
    data_train["preprocessed_text"]
)

# Convert the VALIDATION data using the SAME vocabulary
# and the SAME weights learned from the training data.
X_val_tfidf = tfidf_vectorizer.transform(
    data_val["preprocessed_text"]
)

# Print the dimensions of the vectorized datasets.
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)

Training TF-IDF shape: (800, 28042)
Validation TF-IDF shape: (200, 28042)


## And the Train a Classifier?

In [20]:
# Your code

# Import the Multinomial Naive Bayes classifier.
# This model is commonly used for text classification
# because it works well with word-count and TF-IDF features.
from sklearn.naive_bayes import MultinomialNB

# Import accuracy_score so we can measure
# how many validation predictions are correct.
from sklearn.metrics import accuracy_score


# Create the classifier.
# We use the default parameters, as required by the lab.
model = MultinomialNB()


# Train the classifier using the TF-IDF training features
# and the correct labels.
model.fit(
    X_train_tfidf,
    y_train
)


# Ask the trained model to predict the labels
# for the validation emails.
y_pred = model.predict(
    X_val_tfidf
)


# Compare the predictions with the real validation labels.
accuracy = accuracy_score(
    y_val,
    y_pred
)


# Print the result.
print("Validation accuracy:", accuracy)

Validation accuracy: 0.935


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [21]:
# Your code

# Import a tool for combining sparse text features
# with the additional numeric features.
from scipy.sparse import hstack

# --------------------------------------------------
# 1. BAG OF WORDS ONLY
# --------------------------------------------------

# Create a new Multinomial Naive Bayes model.
model_bow = MultinomialNB()

# Train the model using Bag of Words features.
model_bow.fit(
    X_train_bow,
    y_train
)

# Predict the validation labels.
pred_bow = model_bow.predict(
    X_val_bow
)

# Measure the validation accuracy.
accuracy_bow = accuracy_score(
    y_val,
    pred_bow
)


# --------------------------------------------------
# 2. TF-IDF ONLY
# --------------------------------------------------

# Create another fresh model.
model_tfidf = MultinomialNB()

# Train using TF-IDF features.
model_tfidf.fit(
    X_train_tfidf,
    y_train
)

# Predict the validation labels.
pred_tfidf = model_tfidf.predict(
    X_val_tfidf
)

# Measure the validation accuracy.
accuracy_tfidf = accuracy_score(
    y_val,
    pred_tfidf
)


# --------------------------------------------------
# 3. PREPARE THE EXTRA FEATURES
# --------------------------------------------------

# Select the three extra numeric features
# that were created earlier in the notebook.
extra_train = data_train[
    ["money_mark", "suspicious_words", "text_len"]
]

extra_val = data_val[
    ["money_mark", "suspicious_words", "text_len"]
]


# --------------------------------------------------
# 4. BAG OF WORDS + EXTRA FEATURES
# --------------------------------------------------

# Combine the Bag of Words matrix
# with the three extra numeric columns.
X_train_bow_extra = hstack([
    X_train_bow,
    extra_train.values
])

X_val_bow_extra = hstack([
    X_val_bow,
    extra_val.values
])

# Create a fresh model.
model_bow_extra = MultinomialNB()

# Train using BoW + extra features.
model_bow_extra.fit(
    X_train_bow_extra,
    y_train
)

# Predict the validation labels.
pred_bow_extra = model_bow_extra.predict(
    X_val_bow_extra
)

# Measure the accuracy.
accuracy_bow_extra = accuracy_score(
    y_val,
    pred_bow_extra
)


# --------------------------------------------------
# 5. TF-IDF + EXTRA FEATURES
# --------------------------------------------------

# Combine TF-IDF features
# with the same three extra numeric columns.
X_train_tfidf_extra = hstack([
    X_train_tfidf,
    extra_train.values
])

X_val_tfidf_extra = hstack([
    X_val_tfidf,
    extra_val.values
])

# Create another fresh model.
model_tfidf_extra = MultinomialNB()

# Train using TF-IDF + extra features.
model_tfidf_extra.fit(
    X_train_tfidf_extra,
    y_train
)

# Predict the validation labels.
pred_tfidf_extra = model_tfidf_extra.predict(
    X_val_tfidf_extra
)

# Measure the accuracy.
accuracy_tfidf_extra = accuracy_score(
    y_val,
    pred_tfidf_extra
)


# --------------------------------------------------
# 6. COMPARE ALL FOUR RESULTS
# --------------------------------------------------

print("Bag of Words only:", accuracy_bow)
print("TF-IDF only:", accuracy_tfidf)
print("Bag of Words + extra features:", accuracy_bow_extra)
print("TF-IDF + extra features:", accuracy_tfidf_extra)

Bag of Words only: 0.945
TF-IDF only: 0.935
Bag of Words + extra features: 0.81
TF-IDF + extra features: 0.555
